# TCML CNP zero-shot, resonance block withheld

Generalisation test: the whole resonance band, E in [0.08, 0.16], is withheld from training (data/split_block.csv).

Leak-free inputs, defaults of the trainer: `data/combined_data_trainval.csv` (training and validation
elasticities only) and `data/branch_functional_descriptors_leakfree_trainval.csv`.


In [ ]:
import subprocess, sys
# the stock torch of the image does not support every GPU of the fleet;
# this needs 'Internet' enabled in the notebook settings
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                      '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())


In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
# the dataset attached to this notebook must be this repository, data/ folder included
REPO_SRC = list(INPUT.rglob('train/train_star_pro.py'))[0].parents[1]
REPO = Path('/kaggle/working/TaylorCouetteML')
if REPO.exists(): shutil.rmtree(REPO)
shutil.copytree(REPO_SRC, REPO)
os.chdir(REPO)
print('repository ready at', REPO)


In [ ]:
OUT = Path('/kaggle/working/runs/cnp_lf_block'); OUT.mkdir(parents=True, exist_ok=True)
EPOCHS, BATCH, LR, N_SEEDS = 1200, 16, 2e-4, 5
args = [sys.executable, 'train/train_cnp_leakfree.py', '--epochs', str(EPOCHS), '--batch', str(BATCH), '--lr', str(LR), '--n_seeds', str(N_SEEDS), '--ctx_noise', '0.03', '--max_ctx_points', '32', '--p_zero_shot', '0.80', '--n_cheb', '64', '--n_cos', '32', '--d_model', '256', '--n_enc_layers', '6', '--n_dec_layers', '3', '--n_heads', '8', '--n_k_freq', '16', '--lam_spec', '5e-5', '--lam_smooth', '5e-4', '--margin', '0.05', '--split_csv', 'data/split_block.csv', '--out_root', str(OUT)]
print('>>>', ' '.join(args)); t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO))
print('<<< exit', rc, 'elapsed', round((time.time() - t0) / 60, 1), 'min')
assert rc == 0, 'training failed'


In [ ]:
for root, dirs, files in os.walk(OUT):
    for f in sorted(files):
        p = Path(root) / f
        print(p.relative_to(OUT), round(p.stat().st_size / 1e6, 1), 'MB')
